In [ ]:
# ==========================================
# Colab：用“原始EMG/Slope”做平台折线（不改原始形态）
# - dt_sec：忽略日期，只用 HH:MM:SS；跨午夜自动续接
# - 平台折线：在“原始值”上取峰/谷（保留向上峰 + 向下谷）
# - 输出：
#   1) combined_plateau_raw.png              黑底双轨（图3风格）
#   2) check_emg_overlay.png                 对照：原始EMG(灰) + 简化EMG(蓝)
#   3) layer_emg.png / layer_slope.png / layer_legend_axes.png   三层透明PNG
# ==========================================

!pip -q install pandas openpyxl numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from google.colab import files

# -----------------------------
# 0) 上传并读取（csv/xlsx）
# -----------------------------
print("请上传文件（csv 或 xlsx），需包含：dt_sec, emg_raw，以及 slope 列（slope_cps / slope_val / slope*）")
up = files.upload()
fn = next(iter(up.keys()))

def read_any(fn: str) -> pd.DataFrame:
    if fn.lower().endswith(".csv"):
        try:
            return pd.read_csv(fn)
        except UnicodeDecodeError:
            return pd.read_csv(fn, encoding="utf-8-sig")
    return pd.read_excel(fn)

df = read_any(fn)
print("Loaded:", fn)
print("Columns:", list(df.columns))

# -----------------------------
# 1) 自动识别列名
# -----------------------------
def pick_col(df_cols, exact_candidates, contains_keys=None):
    for c in exact_candidates:
        if c in df_cols:
            return c
    if contains_keys:
        for c in df_cols:
            low = c.lower()
            if any(k in low for k in contains_keys):
                return c
    return None

cols = list(df.columns)
time_col  = pick_col(cols, ["dt_sec", "time", "timestamp"], contains_keys=["dt", "time"])
emg_col   = pick_col(cols, ["emg_raw", "emg"], contains_keys=["emg"])
slope_col = pick_col(cols, ["slope_cps", "slope_val"], contains_keys=["slope"])

if time_col is None:  raise ValueError(f"找不到时间列 dt_sec。当前列：{cols}")
if emg_col is None:   raise ValueError(f"找不到 EMG 列 emg_raw。当前列：{cols}")
if slope_col is None: raise ValueError(f"找不到 slope 列（slope_cps/slope_val/slope*）。当前列：{cols}")

print("识别到列：", {"time_col": time_col, "emg_col": emg_col, "slope_col": slope_col})

work = df[[time_col, emg_col, slope_col]].copy()

# -----------------------------
# 2) dt_sec -> unwrap 秒数（保留真实钟表时间；跨午夜续接）
# -----------------------------
def parse_dtsec_to_unwrap_seconds_keep_clock(s: pd.Series) -> np.ndarray:
    s_str = s.astype(str)

    # 抽取 HH:MM:SS(.ms)
    m = s_str.str.extract(r'(\d{1,2}:\d{2}:\d{2}(?:\.\d+)?)', expand=False)
    t1 = pd.to_datetime(m, errors="coerce")

    t_sec = np.full(len(s_str), np.nan, dtype=float)
    ok1 = t1.notna()
    if ok1.any():
        t_sec[ok1.to_numpy()] = (
            t1.dt.hour[ok1] * 3600 +
            t1.dt.minute[ok1] * 60 +
            t1.dt.second[ok1]
        ).to_numpy(dtype=float)

    # 数字兜底：Excel序列/纯数字
    left = np.isnan(t_sec)
    if left.any():
        num = pd.to_numeric(s[left], errors="coerce")
        frac = num - np.floor(num)
        sec_from_excel = frac * 86400
        sec_from_mod = np.mod(num, 86400)
        use_excel = frac.notna() & (frac > 0)
        t_sec[left] = np.where(use_excel, sec_from_excel, sec_from_mod)

    nan_ratio = np.isnan(t_sec).mean()
    if nan_ratio > 0.5:
        bad = s_str[np.isnan(t_sec)].head(10).tolist()
        raise ValueError(f"{time_col} 解析失败太多（{nan_ratio:.1%}）。示例：{bad}")

    # 跨午夜解包
    t_int = np.round(t_sec).astype(int)
    t_unwrap = t_int.copy()
    for i in range(1, len(t_unwrap)):
        if t_unwrap[i] < t_unwrap[i-1]:
            t_unwrap[i:] += 86400

    return t_unwrap

work["_t"] = parse_dtsec_to_unwrap_seconds_keep_clock(work[time_col])

# 数值化 + 排序
work[emg_col] = pd.to_numeric(work[emg_col], errors="coerce")
work[slope_col] = pd.to_numeric(work[slope_col], errors="coerce")
work = work.dropna(subset=["_t", emg_col, slope_col]).sort_values("_t")

# -----------------------------
# 3) 每秒聚合（同一秒多行取均值；不插值补缺秒，避免“形态变平”）
# -----------------------------
sec = work["_t"].astype(int)
agg = work.groupby(sec)[[emg_col, slope_col]].mean().sort_index()
x = agg.index.to_numpy(dtype=float)

emg_raw = agg[emg_col].to_numpy(dtype=float)
slope_raw = agg[slope_col].to_numpy(dtype=float)

# -----------------------------
# 4) 平台折线：在“原始值”上取峰/谷（保留向上峰 + 向下谷）
#    只用轻度平滑(找极值用)，平台高度=原始y[i]
# -----------------------------
def winsorize(y, q_low=0.005, q_high=0.995):
    lo = np.nanquantile(y, q_low)
    hi = np.nanquantile(y, q_high)
    return np.clip(y, lo, hi)

def find_extrema_indices(y_smooth: np.ndarray) -> np.ndarray:
    dy = np.diff(y_smooth)
    sgn = np.sign(dy)
    for i in range(1, len(sgn)):
        if sgn[i] == 0:
            sgn[i] = sgn[i-1]
    ext = []
    for i in range(1, len(sgn)):
        if sgn[i-1] > 0 and sgn[i] < 0:
            ext.append(i)   # 峰
        elif sgn[i-1] < 0 and sgn[i] > 0:
            ext.append(i)   # 谷
    return np.array(ext, dtype=int)

def keep_extrema_minsep(ext_idx, y_ref, min_sep=4):
    if len(ext_idx) == 0:
        return ext_idx
    ext_idx = np.array(sorted(ext_idx), dtype=int)
    keep = [ext_idx[0]]
    for idx in ext_idx[1:]:
        if idx - keep[-1] >= min_sep:
            keep.append(idx)
        else:
            # 同一窗口里留“变化更大”的（峰或谷都可能）
            prev = keep[-1]
            if abs(y_ref[idx] - np.median(y_ref)) > abs(y_ref[prev] - np.median(y_ref)):
                keep[-1] = idx
    return np.array(keep, dtype=int)

def plateau_polyline_from_raw(x, y_raw, smooth_win=5, min_sep=4, hold=2,
                              clip_q=(0.005, 0.995)):
    y_raw = np.asarray(y_raw, dtype=float)

    # ✅ 只用于找极值的“参考序列”：轻度裁剪+轻度平滑
    y_ref = winsorize(y_raw, clip_q[0], clip_q[1])
    y_smooth = pd.Series(y_ref).rolling(smooth_win, center=True, min_periods=1).mean().to_numpy()

    ext = find_extrema_indices(y_smooth)
    if len(ext) < 2:
        return np.array(x), np.array(y_raw)

    ext = keep_extrema_minsep(ext, y_smooth, min_sep=min_sep)

    anchors = np.array(sorted(set([0] + ext.tolist() + [len(y_raw) - 1])), dtype=int)

    half = max(1, hold // 2)
    xp, yp = [x[anchors[0]]], [y_raw[anchors[0]]]

    for k in range(1, len(anchors)-1):
        i = anchors[k]
        a = max(0, i - half)
        b = min(len(y_raw) - 1, i + half)

        level = y_raw[i]  # ✅ 平台高度用“原始值”
        xp += [x[a], x[b]]
        yp += [level, level]

    xp.append(x[anchors[-1]])
    yp.append(y_raw[anchors[-1]])
    return np.array(xp), np.array(yp)

# --------- 你可以调这几个参数（只影响“简化程度”，不改原始数据）---------
SMOOTH_WIN_EMG = 5   # 越小越敏感（更密）
SMOOTH_WIN_SLO = 5
MIN_SEP        = 4   # 越小峰谷越多（2~6）
HOLD           = 2   # 平台长度（1~3）
CLIP_Q         = (0.005, 0.995)

emg_xp, emg_yp = plateau_polyline_from_raw(
    x, emg_raw, smooth_win=SMOOTH_WIN_EMG, min_sep=MIN_SEP, hold=HOLD, clip_q=CLIP_Q
)
slope_xp, slope_yp = plateau_polyline_from_raw(
    x, slope_raw, smooth_win=SMOOTH_WIN_SLO, min_sep=MIN_SEP, hold=HOLD, clip_q=CLIP_Q
)

# -----------------------------
# 5) 时间格式（%86400 显示钟表时间）
# -----------------------------
def fmt_hms_unwrap(v, pos=None):
    v = int(round(v))
    v_mod = v % 86400
    h = v_mod // 3600
    m = (v_mod % 3600) // 60
    s = v_mod % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

# -----------------------------
# 6) 对照图：原始EMG(灰) + 简化EMG(蓝) —— 用来确认“对得上”
# -----------------------------
plt.figure(figsize=(18, 3.2), dpi=160)
axc = plt.gca()
axc.set_facecolor("white")
plt.gcf().patch.set_facecolor("white")

axc.plot(x, emg_raw, linewidth=0.8, alpha=0.45)          # 原始
axc.plot(emg_xp, emg_yp, linewidth=1.6, alpha=0.95)      # 简化

axc.xaxis.set_major_formatter(FuncFormatter(fmt_hms_unwrap))
axc.xaxis.set_major_locator(MaxNLocator(nbins=10))
axc.grid(True, axis="x", alpha=0.25)
axc.set_title("EMG raw (gray) vs plateau simplified (blue)")

plt.tight_layout()
chk = "check_emg_overlay.png"
plt.savefig(chk, bbox_inches="tight")
plt.show()
print("已输出对照图：", chk)
files.download(chk)

# -----------------------------
# 7) 黑底双轨（图3风格）+ 三层导出
#    为了同图显示：对 EMG / slope 各自做“线性显示缩放”，但不改形态
# -----------------------------
def display_scale(y, q=0.98):
    """
    仅用于把不同量纲压到类似高度（不改形态，不做正负翻转）
    """
    y = np.asarray(y, dtype=float)
    span = np.nanquantile(np.abs(y - np.nanmedian(y)), q) + 1e-9
    return (y - np.nanmedian(y)) / span  # 只做线性中心化+缩放（形态保留）

emg_disp   = display_scale(emg_yp, q=0.98)
slope_disp = display_scale(slope_yp, q=0.98)

# 把 x 也用同样的 plateau x
fig = plt.figure(figsize=(18, 3.8), dpi=160)
ax = plt.gca()
ax.set_facecolor("black")
fig.patch.set_facecolor("black")

emg_offset   = 1.2
slope_offset = -1.2

emg_line, = ax.plot(emg_xp,   emg_disp   + emg_offset,   linewidth=1.8, alpha=0.95)
slope_line, = ax.plot(slope_xp, slope_disp + slope_offset, linewidth=1.8, alpha=0.95)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("white")
ax.tick_params(axis="x", colors="white", labelsize=8)
ax.tick_params(axis="y", left=False, labelleft=False)

ax.xaxis.set_major_formatter(FuncFormatter(fmt_hms_unwrap))
ax.xaxis.set_major_locator(MaxNLocator(nbins=12))
ax.grid(True, axis="x", alpha=0.12)

ymin = min((emg_disp + emg_offset).min(), (slope_disp + slope_offset).min())
ymax = max((emg_disp + emg_offset).max(), (slope_disp + slope_offset).max())
ax.set_ylim(ymin - 1.2, ymax + 1.2)

label_box = dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.9)
t1 = ax.text(0.06, 0.30, "EMG (plateau from RAW)", transform=ax.transAxes, color="black",
             fontsize=9, va="center", ha="left", bbox=label_box)
t2 = ax.text(0.06, 0.22, f"{slope_col} (plateau from RAW)", transform=ax.transAxes, color="black",
             fontsize=9, va="center", ha="left", bbox=label_box)

plt.tight_layout()

# 整图输出
out_full = "combined_plateau_raw.png"
fig.savefig(out_full, facecolor="black", bbox_inches="tight")
plt.show()
print("已输出整图：", out_full)
files.download(out_full)

# ---------- 三层导出 ----------
def set_axes_artists_visible(ax, visible: bool):
    for spine in ax.spines.values():
        spine.set_visible(visible)
    ax.xaxis.set_visible(visible)
    ax.yaxis.set_visible(visible)
    for gl in ax.get_xgridlines() + ax.get_ygridlines():
        gl.set_visible(visible)

def save_layer(filename, show_emg, show_slope, show_axes_legend):
    emg_line.set_visible(show_emg)
    slope_line.set_visible(show_slope)
    t1.set_visible(show_axes_legend)
    t2.set_visible(show_axes_legend)
    set_axes_artists_visible(ax, show_axes_legend)
    fig.savefig(filename, transparent=True, bbox_inches="tight", pad_inches=0.02)

save_layer("layer_emg.png", show_emg=True,  show_slope=False, show_axes_legend=False)
save_layer("layer_slope.png", show_emg=False, show_slope=True,  show_axes_legend=False)
save_layer("layer_legend_axes.png", show_emg=False, show_slope=False, show_axes_legend=True)

print("已输出三层：layer_emg.png, layer_slope.png, layer_legend_axes.png")
files.download("layer_emg.png")
files.download("layer_slope.png")
files.download("layer_legend_axes.png")

print("\n参数怎么调（只改算法，不改原始数据）：")
print("MIN_SEP 变小(2~3) => 峰谷更多、更密；变大(5~8) => 更简化")
print("SMOOTH_WIN_* 变小(3) => 更敏感；变大(7~9) => 更稳更平滑")
print("HOLD 变小(1) => 平台更短；变大(3) => 平台更长")